In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [2]:
# dummy dataset
dataset = [
    ["# 0 + 1 # 1 - 0 <eos>", "<sos> # 1 + 2 # 2 - 1", "# 1 + 2 # 2 - 1 <eos>"],
    ["# 0 + 1 # 1 - 0 # 1 + 2 # 2 - 1 <eos>", "<sos> # 2 + 3 # 3 - 2", "# 2 + 3 # 3 - 2 <eos>"],
]
vocab = ["<sos>", "<eos>", "#", "+", "-", "0", "1", "2", "3"]

In [ ]:
# if decoder's forward() is called without specifying the hidden state, then hidden state is initialized to zeros
# https://github.com/pytorch/pytorch/blob/v2.7.0/torch/nn/modules/rnn.py#L1053 

In [5]:
def tensorize_input(s, vocab):
    return torch.tensor([vocab.index[sym] for sym in s.split()], dtype=torch.long)


In [7]:
class Seq2Seq(nn.Module):
    def __init__(self, vocab, embedding_dim, hidden_dim):
        super(Seq2Seq, self).__init__()
        self.sym2idx = {sym: idx for idx, sym in enumerate(vocab)}
        self.idx2sym = {idx: sym for idx, sym in enumerate(vocab)}
        self.embedding = nn.Embedding(num_embeddings=len(vocab), embedding_dim=embedding_dim)
        self.encoder = nn.LSTM(embedding_dim, hidden_dim)
        self.decoder = nn.LSTM(embedding_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, len(vocab))

    def forward(self, batched_enc_input, batched_dec_input):
        batched_enc_embeddings = self.embedding(batched_enc_input)
        batched_dec_embeddings = self.embedding(batched_dec_input)

        _, enc_hidden = self.encoder(batched_enc_embeddings)
        dec_output, _ = self.decoder(batched_dec_embeddings, enc_hidden)
        output = self.fc(dec_output.squeeze)
        return output

In [8]:
#batched_dec_output
embedding_dim = 128
hidden_size = 256

model = Seq2Seq(vocab, embedding_dim, hidden_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
enc_input_tensors = []
dec_input_tensors = []
dec_output_tensors = []
for example in dataset:
    enc_input = tensorize_input(example[0], vocab)
    dec_input = tensorize_input(example[1], vocab)
    dec_output = tensorize_input(example[2], vocab)
    
    enc_input_tensors.append(enc_input)
    dec_input_tensors.append(dec_input)
    dec_output_tensors.append(dec_output)

In [ ]:
epochs = 10
